# MMA basics tutorial

This notebook is designed to teach the basic operation of the main MMA computaitonal pipeline. It will first show how to prepare the input file for all parts of the model. Then, it runs the computational pipeline from bash. Finally, we analyse the results. It follows the same logic as the more advanced tutorials, where these steps are run independently. It is recommended to get familiar (at least conceptually) with all the steps in this tutorial. The key steps of the tutorial are
1) [Definition of the physical parameters of the simulation](#physical_parameters);
2) [Definition of the numerical parameters of the simulation](#numerical_parameters);
3) [Definition of the numerical parameters of the simulation](#prepare_the_input_file);
4) [Run the simulation in terminal](#run_the_pipeline);
5) [Analyse the results](#analyse_the_results).

The parameters natively in this simmulation are very short propagation and only a few microscopic responses such that it runs in a reasonalbe time even on few cores. If the testing local machine provides a mediocore number of cores (>20), even a small physically meaningful simulation can be run.

The first cell just loads the python modules.

In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import shutil
import h5py
import sys
import MMA_administration as MMA
import mynumerics as mn
import units
import HHG
from IPython.display import display, Markdown, HTML


%matplotlib inline
# import mpld3
# mpld3.enable_notebook()

Here we specify the path, where we will store our data. It is created in the work directory. You have an automatically opened terminal in the other tab. Try to switch to this path alse there by `cd $MULTISCALE_WORK_DIR/mma_basics`.

In [ ]:
outputs_path = os.path.join(os.environ['MULTISCALE_WORK_DIR'],'mma_basics')

## Physical parameters
<a id="physical_parameters"></a>
<a id="reference_Gaussian"></a>
The philosophy of our input uses a "reference Gaussian beam". This means that our reference is the Gaussian beam with known parameters (focus, waist, focus intensity) propagating in vacuum. Then we add a medium in the path of the beam in our experiment. Finally, we specify the parameters of the XUV camera.

### Medium parameters
First we specify the medium. It uses pre-defined material constants for rare gases (`He`, `Ne`, `Ar`, `Kr`, `Xe`). There is also specified the ionisation model (`PPT`) and the dispersion and absorption relation in the XUV range (`NIST` and `Henke` tables are available).$^\dagger$

$^\dagger$ Be aware that these tables might be limited for soft-XUV (low harmonics $\Leftrightarrow$ long wavelenghts).

In [ ]:
# gas specifiers
gas = 'Ar'
medium_length =   40e-6 # [m]
medium_pressure = 100e-3 # [mbar]

# sources of further "material constants"
ionisation_model = 'PPT'
XUV_dispersion_tables = 'NIST'
XUV_absorption_tables = 'Henke'

### Laser parameters
We specify the input beam using the aforementioned reference Gaussian beam. Here we scan in the peak input intensities, all the other parameters remains the same.

We show the peak intensity inferred from inverting the formula $E_{\text{cutoff}} = I_p + 3.17 U_p$, which is implemented in the HHG module. This could be convenient for some HHG studies. The 

In [ ]:
laser_wavelength = 800e-9 # [m]
reference_Gaussian_focus = 0.
reference_Gaussian_waist = 20e-6 # [m]
laser_pulse_duration = 10e-15 # [s] (defined via 1/e in the electric field amplitude)

# This function inverts the selected harmonic into the laser intensity.
# It operates in atomic units, so there is the conversion factor to SI.
peak_intensity = HHG.ComputeInvCutoff_gas(20.,mn.ConvertPhoton(laser_wavelength,'lambdaSI','omegaau'),gas = gas)*units.INTENSITYau # [W/m2]
# peak_intensity = 8.0e17 # It can be defined also in SI directly

### XUV camera
The XUV camera is specified by the respective position to the cell, spectral width, and the radial size. (The resolutions are defined as Numerical parameters.)

In [ ]:
XUV_camera_distance         = 1.                      # [m] (from the entry of the cell)
XUV_camera_harmonic_range   = np.asarray([14., 28.])  # [harmonic order]
XUV_camera_radial_range     = 0.007                   # [m]

## Numerical parameters

<a id="numerical_parameters"></a>
Here we define the numerical parameters. This release of the code leaves the responsibility of choosing proper parameters to users (some heuristics are below), except the implementatation of adaptive steps in $z$.

### CUPRAD (pulse propagation)

In [ ]:
number_of_points_in_r      = 256
number_of_points_in_t      = 256

operators_t                = 2
first_delta_z              = 0.01 # [mm]
phase_threshold_for_decreasing_delta_z = 0.002	# [rad]

length_of_window_for_r_normalized_to_beamwaist = 4.   # [-]
length_of_window_for_t_normalized_to_pulse_duration = 4. # [-]

number_of_absorber_points_in_time = 16  # [-]

physical_output_distance_for_plasma_and_Efield = 10e-6   # [m]

output_distance_in_z_steps_for_fluence_and_power   = 100  # [-]

radius_for_diagnostics = 0.1 # [cm]

run_time_in_hours = 5.0 # [h] 

In [ ]:
## Code to generate the following text ##
zR = (np.pi*reference_Gaussian_waist**2)/laser_wavelength
dr_CUPRAD = length_of_window_for_r_normalized_to_beamwaist * reference_Gaussian_waist*np.sqrt(1+(reference_Gaussian_focus/zR)**2)/number_of_points_in_r
display(Markdown(rf"""### Properties of the chosen discretisation
* The chosen discretisation in time gives ~ {
            number_of_points_in_t/(
            laser_pulse_duration*length_of_window_for_t_normalized_to_pulse_duration/mn.ConvertPhoton(laser_wavelength,'lambdaSI','T0SI')
            )
    :.0f}
points per one laser period.
* The stepsize in the radial discretisation is ~ ${
      1e6*dr_CUPRAD
      :.2f}
~\mu {{\mathrm{{m}}}}$.
* The size of the radial computational box is ~ ${
      1e6*length_of_window_for_r_normalized_to_beamwaist * reference_Gaussian_waist
      :.2f}
~\mu {{\mathrm{{m}}}}$. The maximal radius of the reference Gaussian beam is ~ ${
      1e6*np.max([
            reference_Gaussian_waist*np.sqrt(1+((medium_length-reference_Gaussian_focus)/zR)**2),
            reference_Gaussian_waist*np.sqrt(1+(reference_Gaussian_focus/zR)**2)
            ])
      :.2f}
~\mu {{\mathrm{{m}}}}$.$^\dagger$ 

$^\dagger$ This is given at the $z$-edges of the copmutational box.
"""))

### TDSE
Here we specify the computational grids and other numerical parameters. The macroscopic grid (where the TDSE's are computed in the macroscopic volume) is a subgrid of the CUPRAD grid and is specified by strides. The microscopic grids used by the TDSE's computational routines are specified by the time and space steps and the numebr of steps in space (time is inherited from CUPRAD).

*Further insight in the results is discussed in the tutorial on TDSE `teach-me-tdse`. Based on experiments in [this notebook](../interactive_TDSE/field_from_CUPRAD.ipynb), there are found conditions to "filter long trajectories". Try to uncomment the absorbers together with reducing the spatial extent `Nx_max` and re-run the simulations after you analyse the whole notebook.*

In [ ]:
# Macroscopic: to slect the grid based on the CUPRAD grid
kz_step = 1
kr_step_CTDSE = 4
Nr_max = 110

# Microscopic part
dt_TDSE = 0.25 # [a.u.]
dx      = 0.4  # [a.u.]
# Nx_max  = 7000 # (spans from -dx*Nx_max to dx*Nx_max) 
Nx_max  = 150 # try with absorber
x_int   = 2.0  # [a.u.]     # "Microscopic volume of the atom." Used to define the volumetric ionisation.

# absorber
# absorber_type  = 0;  # No absorber

# Complex absorber
absorber_type  = 1;
absorber_x_cap = 50.;
absorber_alpha = 0.001;

CV_criterion_of_ground_state = 1e-12 # [-]

# Outputs
# choose from: 'electric field', 'electric field (Fourier),
#              '<dj/dt>', '<dj/dt> (Fourier)'
#              'ground-state population (projected)', 'ground-state population (integrated)'
#              '<x>'
list_of_CTDSE_outputs = ['electric field', '<dj/dt> (Fourier)',
                         'ground-state population (projected)', '<x>']

In [ ]:
## Code to generate the following text ##

reference_Gaussian_focus_intensity = peak_intensity
onax_entry_intensity = (reference_Gaussian_focus_intensity/units.INTENSITYau)/np.sqrt(1+(reference_Gaussian_focus/zR)**2)
onax_entry_ponderomotive_potential = onax_entry_intensity/(4.*mn.ConvertPhoton(laser_wavelength,'lambdaSI','omegaau'))
max_Ek_direct = 5.*onax_entry_ponderomotive_potential
max_electron_velocity_direct = np.sqrt(2.*max_Ek_direct)
max_Ek_rescattered = 10.*onax_entry_ponderomotive_potential
max_electron_velocity_rescattered = np.sqrt(2.*max_Ek_rescattered)
t_box = (length_of_window_for_t_normalized_to_pulse_duration*laser_pulse_duration)/units.TIMEau

display(Markdown(rf"""### Physical consequences of the chosen numerical parameters
* The step-size in $z$ is derived from the CUPRAD's adaptive steps, the stride for CTDSE is {kz_step}. (Generally, we do not recommend to use stride > 1.$^\dagger$)
* The macroscopic radial discretisation for TDSE is ${
      1e6*kr_step_CTDSE*dr_CUPRAD  
      :.2f}
~\mu {{\mathrm{{m}}}}$; the macroscopic radial boxsize is $r_{{\text{{max}}}}={
    1e6*Nr_max*dr_CUPRAD
    :.2f}
~\mu {{\mathrm{{m}}}}$.
* The outputs stored from CTDSE runs are: {', '.join(['***'+foo+'***' for foo in list_of_CTDSE_outputs])}.
* Possible ouputs *ground-state population (projected)* and *ground-state population (integrated)* refer to various approaches to ionisation. See
[link 1](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.106.053115) or [link 2](https://theses.hal.science/tel-04192431v1/document)$^{{\dagger\dagger}}$ (Chapter 3) for details.
* The microscopic computatinal box for 1D-TDSE is $x_{{\text{{max}}}} = {
    dx*Nx_max
    :.0f}~{{\mathrm{{a.u.}}}}~({
    1e9*dx*Nx_max * units.LENGTHau    
    :.2f}~{{\mathrm{{nm}}}})$.
* The maximal energy in the spectrum according to the chosen discretisation is $E_{{\text{{max}}}} = {
    mn.ConvertPhoton((2.*np.pi/dt_TDSE),'omegaau','eV')
    :.2f}~{{\mathrm{{eV}}}}$ ($H_{{\text{{max}}}} \sim {
    (2.*np.pi/dt_TDSE)/mn.ConvertPhoton(laser_wavelength,'lambdaSI','omegaau')
    :.0f}$).
* Theoretical maximal distances of a classical electron ejected at the peak of the pulse reached at,
respectively, the end of the compuational box and the trailing edge of the pulse ($1/\mathrm{{e}}^2$ of the intensity).$^{{\dagger\dagger\dagger}}$
    * [Direct electrons with $E_{{\text{{kin}}}} \sim 5U_p$](https://doi.org/10.1103/PhysRevA.106.053115) $s_{{\text{{max}}}} = {
    max_electron_velocity_direct*0.5*t_box
    :.2f}~{{\mathrm{{a.u.}}}}~({
    1e9*max_electron_velocity_direct*0.5*t_box*units.LENGTHau    
    :.2f}~{{\mathrm{{nm}}}})$.
    * [Rescatterred electrons $E_{{\text{{kin}}}} \sim 10U_p$](https://doi.org/10.1038/nphys914) $s_{{\text{{max}}}} = {
    max_electron_velocity_rescattered*0.5*t_box
    :.2f}~{{\mathrm{{a.u.}}}}~({
    1e9*max_electron_velocity_rescattered*0.5*t_box*units.LENGTHau    
    :.2f}~{{\mathrm{{nm}}}})$, $s_{{\text{{max,2}}}} = {
    max_electron_velocity_rescattered*0.5*(laser_pulse_duration/units.TIMEau)
    :.2f}~{{\mathrm{{a.u.}}}}~({
    1e9*max_electron_velocity_rescattered*0.5*(laser_pulse_duration/units.TIMEau)*units.LENGTHau    
    :.2f}~{{\mathrm{{nm}}}})$.
* The cut-off for the peak on-axis entry intensity is $H_{{\text{{cut-off}}}} = {
   HHG.ComputeCutoff_gas(onax_entry_intensity,mn.ConvertPhoton(laser_wavelength,'lambdaSI','omegaau'),gas=gas)[1]
    :.2f}$ (given by $I_P + 3.17 U_p$).


$^\dagger$ The step-size in $z$ is usually adapted such that the variations of the phase are affecting XUV when using a coarser grid. \
$^{{\dagger\dagger}}$ *ground-state population (projected)* corresponds to Eq. (3.20) and *ground-state population (projected)* to Eq. (3.1) of [link 2](https://theses.hal.science/tel-04192431v1/document). \
$^{{\dagger\dagger\dagger}}$ This is computed from the theoretical peak intensity. Especially for high intensities, [it might be fastly clamped and the effective peak intensity is lower](https://doi.org/10.1007/s003400100637).
"""))

### Hankel
Here we specify the computational grids for Hankel transoform. These are subgrids of TDSE's grids specified by strides and maxima. Next, we specify by `store_cumulative_field` whether the cumulative results along $z$ are stored. Finally, we chooses the number of threads for the computaiton here. Assuming this example is usually run localy in Docker, we set this number to the numer of available sockets.

In [ ]:
store_cumulative_field          = True
kr_step_Hankel                   = 1
ko_step                          = 1
Nr_max_Hankel_integration        = 256
XUV_camera_number_of_r_points    = 100

Nthreads = int(os.environ['NUM_PROC_DOCKER']) # obtain the number of threads from docker container

In [ ]:
## Code to generate the following text ##
dr_Hankel = kr_step_CTDSE*kr_step_Hankel*dr_CUPRAD


first_diffraction_maximum_cutoff = laser_wavelength/\
        (dr_Hankel*HHG.ComputeCutoff_gas(onax_entry_intensity,mn.ConvertPhoton(laser_wavelength,'lambdaSI','omegaau'),gas=gas)[1])

display(Markdown(rf"""### The role of parameters of the camera and the integration
* The macroscopic radial discretisation for Hankel integral is ${
      1e6*dr_Hankel   
      :.2f}
~\mu {{\mathrm{{m}}}}$; the macroscopic radial boxsize is $r_{{\text{{max, integration}}}}={
    1e6*Nr_max_Hankel_integration*dr_Hankel 
    :.2f}
~\mu {{\mathrm{{m}}}}$.
* The camera size, $r_{{\text{{max, camera}}}}={
    1e3*XUV_camera_radial_range/XUV_camera_distance
    :.2f}
~{{\mathrm{{mm}}}}$, gives the maximal divergence recorded by the XUV is $\theta_{{\text{{max, camera}}}}={
    1e3*np.arctan(XUV_camera_radial_range/XUV_camera_distance ) 
    :.2f}
~{{\mathrm{{mrad}}}}$. (See the initial section with the physical parameters.)
* The diffraction limit for the maximal expected cut-off provided by the discretisation in the integral is  $r_{{\text{{max, camera, cut-off}}}}={
    1e3*first_diffraction_maximum_cutoff
    :.2f}
~{{\mathrm{{mm}}}}$ (corresponding divergence $\theta_{{\text{{max, camera, cut-off}}}}={
    1e3*np.arctan(first_diffraction_maximum_cutoff/XUV_camera_distance ) 
    :.2f}
~{{\mathrm{{mrad}}}}$)
"""))

# print(laser_wavelength/(50.*dr_Hankel))

In [ ]:
# Code to create the input hdf5-file
## First, we prepare dictionaries between hdf5-inputs and this jupyter notebook

global_input_names_to_jupyter_variables = {
    'gas_preset'                                : (np.bytes_(gas),                       '[-]'   ),
    'medium_pressure_in_bar'                    : (medium_pressure,                      '[bar]' )
}


CUPRAD_names_to_jupyter_variables = {
    # laser parameters
    'laser_wavelength'                          : (1e2*laser_wavelength,                  '[cm]'  ),
    'laser_pulse_duration_in_1_e_Efield'        : (1e15*laser_pulse_duration,             '[fs]' ),
    'laser_focus_intensity_Gaussian'            : (reference_Gaussian_focus_intensity,    '[W/m2]'  ),
    'laser_focus_beamwaist_Gaussian'            : (reference_Gaussian_waist,              '[m]'  ),
    'laser_focus_position_Gaussian'             : (reference_Gaussian_focus,              '[m]'  ),

    # medium parameters
    'medium_physical_distance_of_propagation'   : (medium_length,                         '[m]'   ),

    # ionisation
    'ionization_model'                          : (np.bytes_(ionisation_model),          '[-]'  ),

    # numerics
    'numerics_number_of_points_in_r'            : (number_of_points_in_r,                 '[-]'  ),
    'numerics_number_of_points_in_t'            : (number_of_points_in_t,                 '[-]'  ),
    'numerics_operators_t_t-1'                  : (operators_t,                           '[-]'  ),
    'numerics_physical_first_stepwidth'         : (first_delta_z,                         '[mm]' ),
    'numerics_phase_threshold_for_decreasing_delta_z' : 
        (phase_threshold_for_decreasing_delta_z,                '[rad]' ),
    'numerics_length_of_window_for_r_normalized_to_beamwaist':
        (length_of_window_for_r_normalized_to_beamwaist,        '[-]'   ),
    'numerics_length_of_window_for_t_normalized_to_pulse_duration' :
        (length_of_window_for_t_normalized_to_pulse_duration,   '[-]'   ),
    'numerics_number_of_absorber_points_in_time':
        (number_of_absorber_points_in_time ,                    '[-]'   ),
    'numerics_physical_output_distance_for_plasma_and_Efield' :
        (physical_output_distance_for_plasma_and_Efield,        '[m]'   ),
    'numerics_output_distance_in_z-steps_for_fluence_and_power' :
        (output_distance_in_z_steps_for_fluence_and_power,      '[-]'   ),
    'numerics_radius_for_diagnostics'           : (radius_for_diagnostics,                '[cm]' ),
    'numerics_run_time_in_hours'                : (run_time_in_hours,                     '[s]'  )
}


CTDSE_names_to_jupyter_variables = {
    # Physics
    'x_int'                                     : (x_int,                                 '[a.u.]' ),

    # Macro grid
    'Nr_max'                                    : (Nr_max,                                '[-]'    ),
    'kr_step'                                   : (kr_step_CTDSE,                               '[-]'    ),
    'kz_step'                                   : (kz_step,                               '[-]'    ),  

    # Microscopic numerics
    'dx'                                        : (dx,                                    '[a.u.]' ),
    'Nx_max'                                    : (Nx_max,                                '[a.u.]' ),
    'dt'                                        : (dt_TDSE,                               '[a.u.]' ),

    'CV_criterion_of_GS'                        : (CV_criterion_of_ground_state,          '[a.u.]'),
    'absorber_type'  : (absorber_type, '[-]'),
}

# Extend the dictionary if absorbers are used, note different number of inputs is needed for each of them 
if (absorber_type == 1):
    CTDSE_names_to_jupyter_variables = CTDSE_names_to_jupyter_variables|{
        'absorber_alpha' : (absorber_alpha, '[a.u.]'),
        'absorber_x_cap' : (absorber_x_cap, '[a.u.]')    
    }
elif (absorber_type == 2):
    CTDSE_names_to_jupyter_variables['absorber_x_cap'] = (absorber_x_cap, '[a.u.]')

# absorber_type  = 1;
# absorber_x_cap = 20.;
# absirber_alpha = 0.005;


CTDSE_outputs_to_jupyter_names = {
    'print_Efield'                : 'electric field',
    'print_F_Efield'              : 'electric field (Fourier)',
    'print_Source_Term'           : '<dj/dt>',
    'print_F_Source_Term'         : '<dj/dt> (Fourier)',
    'print_GS_population'         : 'ground-state population (projected)',
    'print_integrated_population' : 'ground-state population (integrated)',
    'print_x_expectation_value'   : '<x>'
}

Hankel_names_to_jupyter_variables = {
    'distance_FF'                               : (XUV_camera_distance,                   '[m]'  ),
    'rmax_FF'                                   : (XUV_camera_radial_range,               '[m]'  ),
    'Nr_FF'                                     : (XUV_camera_number_of_r_points,         '[-]'  ),

    'XUV_table_type_dispersion'                 : (np.bytes_(XUV_dispersion_tables),     '[-]'  ),
    'XUV_table_type_absorption'                 : (np.bytes_(XUV_absorption_tables),     '[-]'  ),

    'kr_step'                                   : (kr_step_Hankel,                        '[-]'  ),
    'ko_step'                                   : (ko_step,                               '[-]'  ),
    'Nr_max'                                    : (Nr_max_Hankel_integration,             '[-]'  ),
    'Harmonic_range'                            : (XUV_camera_harmonic_range,      '[harmonic_order]'),

    'store_cumulative_result'                  : (int(store_cumulative_field),          '[-]'  ),   
    'Nthreads'                                  : (Nthreads,                              '[-]'  ) 
}




## Prepare the input file

<a id="prepare_the_input_file"></a>
Here we create the HDF5 file containing all the input parameters. First, we provide several dictionaries (for different modules) to translate the local variables usedin this jupyter notebook to the nomenclature used in the code. Second, we create the list of inputs. It uses two "transformers" of the inputs, which results in inputs either directly in the hdf5-archives or text inputs compatibles with [this parser](https://github.com/vabekjan/universal_input).

We clean up the folder where the inputs are generated, set the filename patterns, and generate the series of the input files.

In [ ]:
## Create inputs

from inputs_transformer import add_variables2hdf5, variables2text

h5_filename        = 'results.h5'
if os.path.exists(outputs_path): shutil.rmtree(outputs_path)  # clean the input directory if it existed
os.makedirs(outputs_path)


h5_filename = os.path.join(outputs_path,h5_filename)

with h5py.File(h5_filename,'w') as f: 

    add_variables2hdf5(f,
                        global_input_names_to_jupyter_variables,
                        CUPRAD_names_to_jupyter_variables,
                        CTDSE_names_to_jupyter_variables,
                        CTDSE_outputs_to_jupyter_names,
                        list_of_CTDSE_outputs,
                        Hankel_names_to_jupyter_variables)
                        


## Run the computational pipeline

<a id="run_the_pipeline"></a>
Now, we have prepared the starting file. Use terminal in the other tab to navigate to the respective directory and verify it. You can also navigate there using the File Browser on the right and inspect the archive.

Now, we run the code. The following cells mimic the bash command line using [bash cell magic](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cell-magics). It is mainly for demonstrative purposes. You can try to run it directly from the terminal in the second tab. Note that we also redirect the ouputs of the codes into log files withour printing them here.

$^\dagger$ It is not easily possible to use the terminal interactively here. So, we need to provide the inputs directly as: `$CUPRAD_BUILD/make_start.e <<< 'results.h5'`. Also, we need to ensure the correct location in each cell by `cd $MULTISCALE_WORK_DIR/mma_basics`. Finally, the behaviour of `mpirun` can prevent the execution of the following commands in the same cell (by establishing MPI environment and redirecting it to MPI rank 0), so we split the cells such that no command follows `mpirun`.

$^\dagger$$^\dagger$ You might see interactively the progression in the logs by e.g. `watch 'cat CUPRAD.log TDSE.log Hankel.log | tail'` in the respective directory.

In [ ]:
%%bash
cd $MULTISCALE_WORK_DIR/mma_basics
$CUPRAD_BUILD/make_start.e <<< 'results.h5'

In [ ]:
%%bash
cd $MULTISCALE_WORK_DIR/mma_basics
mpirun -n "$NUM_PROC_DEFAULT_CUPRAD" $CUPRAD_BUILD/cuprad.e > CUPRAD.log

In [ ]:
%%bash
cd $MULTISCALE_WORK_DIR/mma_basics

python3 $TDSE_1D_PYTHON/prepare_TDSE_Nz.py > TDSE.log

echo "----------------------------------------------" >> TDSE.log
echo "Running TDSE"

mpirun -n $NUM_PROC_DEFAULT_TDSE_1D $TDSE_1D_BUILD/TDSE.e >> TDSE.log

In [ ]:
%%bash
cd $MULTISCALE_WORK_DIR/mma_basics
echo "----------------------------------------------" >> TDSE.log
echo "Merging TDSE results"

python3 $TDSE_1D_PYTHON/merge.py >> TDSE.log

In [ ]:
%%bash
cd $MULTISCALE_WORK_DIR/mma_basics

python3 $HANKEL_HOME/Hankel_long_medium_parallel_cluster.py > Hankel.log

## Analyse the results

<a id="analyse_the_results"></a>
Here we analyse the results directly compared to the other tutorials and usual workflows where the praparatory, execution, and analysing phases are separated.

We provide the basic analyses of the driving field and related plasma density. Next, we show the microscopic spectra, and finally show the far-field distrubution from the Hankel transform. *You can try also open the hdf5 files directly from the browser, which provides a way to quickly review the results. The [jupyter add-on we included](https://github.com/silx-kit/jupyterlab-h5web) allows to visualise interactively most of the results shown below directly.*

Note that the actual version of the code uses independent file `results_Hankel.h5`. The reason is that the Hankel transform can be run repeatedly over the same data (for example with different camera resolution), we do not merge the ouputs into the main archive natively. THe results can be copied to the main file by `python3 $HANKEL_HOME/copy_results_to_main.py` (*try it yourself in terminal*). 

### CUPRAD
The analysis of CUPRAD results uses the Pythonic interface through `dataformat_CUPRAD`, which arranges the outputs into a Pythonic class toghether with various methat that provides natively Fourier transforms of the fiels and other tools.

The following cell then loads the data into the class.

In [ ]:
import dataformat_CUPRAD as dfC
with h5py.File(h5_filename,'r') as f:

    # load cuprad data = pulse propagation
    CUPRAD_res = dfC.get_data(f)    # the class that sotres the CUPRAD outputs
    CUPRAD_res.get_plasma(f)        # plasma is not loaded natively while initialising the class

This defines the limits for plotting. It is not automatised here, so it needs to adjusted to inputs. (*Try yourself to set it automatically to, for example, 1.5 of the pulse pulse duration and spattial extent.*)

In [ ]:
tlim = np.asarray((-15,15))  # [fs]  the time range for plotting the propagating pulse
rlim = 45                   # [mum] the radial range for plotting the propagating pulse

The following code just generates the figure bellow. However, it might be worthy to review the code and check the the way the data are stored:
* The grids `CUPRAD_res.zgrid`, `CUPRAD_res.rgrid`, `CUPRAD_res.tgrid`, `CUPRAD_res.ogrid` (the last is available only once methods for Fourier transform are invoked);
* The data: `CUPRAD_res.E_zrt`, `CUPRAD_res.plasma.value_zrt` (the latter is again obtained as the plasma subclass and generally carries its own grids);
* Various parameters such as `CUPRAD_res.omega0`.

In [ ]:
# Code to generate the animated figure

k_t_min, k_t_max = mn.FindInterval(1e15*CUPRAD_res.tgrid,1.05*tlim)
k_r_max          = mn.FindInterval(1e6*CUPRAD_res.rgrid ,1.05*rlim)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))

r_grid, sym_data = mn.symmetrize_y(1e6*CUPRAD_res.rgrid[:k_r_max],
                    (
                    HHG.ComputeCutoff(
                        mn.FieldToIntensitySI(CUPRAD_res.E_zrt[0,:k_r_max,k_t_min:k_t_max])/units.INTENSITYau,
                        mn.ConvertPhoton(CUPRAD_res.omega0,'omegaSI','omegaau'),
                        mn.ConvertPhoton(CUPRAD_res.Ip_eV,'eV','omegaau')
                    )[1]
                    ).T)

pc1 = ax1.pcolormesh(1e15*CUPRAD_res.tgrid[k_t_min:k_t_max], r_grid, sym_data.T, shading='auto')

ax1.set_xlim(tlim)
ax1.set_ylim((-rlim,rlim))

ax1.set_title("z={:.2f}".format(1e3*CUPRAD_res.zgrid[0]) + ' mm')
ax1.set_xlabel(r'$t~[\mathrm{fs}]$')
ax1.set_ylabel(r'$\rho~[\mu\mathrm{m}]$')

cbar1 = fig.colorbar(pc1, ax=ax1)
cbar1.ax.set_ylabel(r'Intensity [harmonic cut-off]', rotation=90)

r_grid, sym_data = mn.symmetrize_y(1e6*CUPRAD_res.plasma.rgrid[:k_r_max],
                         (1e2/CUPRAD_res.effective_neutral_particle_density)*(CUPRAD_res.plasma.value_zrt[0,:k_r_max,k_t_min:k_t_max]).T
                                    )

pc2 = ax2.pcolormesh(1e15*CUPRAD_res.plasma.tgrid[k_t_min:k_t_max], r_grid, sym_data.T, shading='auto')

cbar2 = fig.colorbar(pc2, ax=ax2)
cbar2.ax.set_ylabel(r'relative plasma density [%]', rotation=90)

ax2.set_xlabel(r'$t~[\mathrm{fs}]$')
ax2.set_ylabel(r'$\rho~[\mu\mathrm{m}]$')


def update(frame):
    # Update the data
    data = (mn.symmetrize_y(1e6*CUPRAD_res.rgrid[:k_r_max], (
            HHG.ComputeCutoff(
                        mn.FieldToIntensitySI(CUPRAD_res.E_zrt[frame,:k_r_max,k_t_min:k_t_max])/units.INTENSITYau,
                        mn.ConvertPhoton(CUPRAD_res.omega0,'omegaSI','omegaau'),
                        mn.ConvertPhoton(CUPRAD_res.Ip_eV,'eV','omegaau')
                    )[1]        
            ).T)[1]).T
    
    # Update the colors
    pc1.set_array(data.ravel())
    pc1.set_clim(data.min(), data.max())
    cbar1.update_normal(pc1)

    ax1.set_title("z={:.2f}".format(1e3*CUPRAD_res.zgrid[frame]) + ' mm')

    data = (mn.symmetrize_y(1e6*CUPRAD_res.plasma.rgrid[:k_r_max],
                          (1e2/CUPRAD_res.effective_neutral_particle_density)*(CUPRAD_res.plasma.value_zrt[frame,:k_r_max,k_t_min:k_t_max]).T
                                    )[1]).T
    
    pc2.set_array(data.ravel())
    pc2.set_clim(data.min(), data.max())
    cbar2.update_normal(pc2)

    return [pc1, pc2]

# Ensure the layout does not have overlaps and everything is nicely spaced
fig.tight_layout() 

ani = matplotlib.animation.FuncAnimation(fig, update, frames=len(CUPRAD_res.zgrid), blit=True)
# ani = matplotlib.animation.FuncAnimation(fig, update, frames=10, blit=True)



plt.close(fig)

HTML(ani.to_jshtml())


These panels show the propagation of the IR-pulse and the correposning plasma density. This is convenient for a cell with a constant pressure. If the dnesity modulation is applied (HHG in gas jet), it might be mor convenient to plot the density directly, see the other tutorials.

### TDSE results analysis
This analysis is similar to CUPRAD. We first set the parameters and then plot the results.

Note that TDSE part is the most memory-consuming part of the code. In larger multiscale workflows, TDSE can be considered as intermediate step and TDSE results are removed after XUV signal is obtained. (Note that HDF5 library requires *repacking* to truly delete data.)

In [ ]:
# Code to create the following figure
kz_analyse = 2       # z-index, where we plot the harmonic spectra
Hmax_plot = 35       # maximal harmonic shown in the plot
rlim_TDSE = 30e-6    # maximal radial extent for the plot

This code is again just to visualise the data. Similarly to CUPRAD, it could be useful to get insight how the data are stored. The paths inside the archive are intentianlly hard-coded, but accessed through the [MMA-helper module](../../shared_python/MMA_administration.py). Note that the organisation is similar also for the low-level codes: [CTDSE](../../1DTDSE/sources/h5namelist.h) and [CUPRAD](../../CUPRAD/sources/global_variables.f90). (*It shall be convenient to use this relative accessing compared to absolute paths to keep the code robust for development.*)

In [ ]:
with h5py.File(h5_filename,'r') as f1:

    k_t_min, k_t_max = mn.FindInterval(1e15*CUPRAD_res.tgrid,1.05*tlim)
    k_r_max          = mn.FindInterval(1e6*CUPRAD_res.rgrid ,1.05*rlim)

    # load TDSE data
    rgrid_TDSE = f1[MMA.paths['CTDSE_outputs'] +'/rgrid_coarse'][:]; Nr_TDSE = len(rgrid_TDSE)
    zgrid_TDSE = f1[MMA.paths['CTDSE_outputs'] +'/zgrid_coarse'][:]; Nz_TDSE = len(zgrid_TDSE)
    ogrid_TDSE = f1[MMA.paths['CTDSE_outputs'] +'/omegagrid'][:]
    tgrid_TDSE = f1[MMA.paths['CTDSE_outputs'] +'/tgrid'][:]
    Hgrid_TDSE = ogrid_TDSE/mn.ConvertPhoton(CUPRAD_res.omega0,'omegaSI','omegaau')



    kz_analyse_TDSE = kz_analyse

    k_r_max_TDSE    = mn.FindInterval(rgrid_TDSE ,rlim_TDSE)

    # it seems that h5py cannot easily provide data for the animation
    spectra_to_plot = [np.abs(    f1[MMA.paths['CTDSE_outputs'] +'/FSourceTerm'][kz_analyse_TDSE,k1,:,0] +
                               1j*f1[MMA.paths['CTDSE_outputs'] +'/FSourceTerm'][kz_analyse_TDSE,k1,:,1])
                       for k1 in range(Nr_TDSE)]

    Efields_to_plot = [f1[MMA.paths['CTDSE_outputs'] +'/Efield'][kz_analyse_TDSE,k1,:] for k1 in range(Nr_TDSE)]



    

    r_grid, sym_data = mn.symmetrize_y(1e6*CUPRAD_res.rgrid[:k_r_max],
                       (
                        HHG.ComputeCutoff(
                            mn.FieldToIntensitySI(CUPRAD_res.E_zrt[kz_analyse,:k_r_max,k_t_min:k_t_max])/units.INTENSITYau,
                            mn.ConvertPhoton(CUPRAD_res.omega0,'omegaSI','omegaau'),
                            mn.ConvertPhoton(CUPRAD_res.Ip_eV,'eV','omegaau')
                        )[1]
                       ).T)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))  # One row, three columns

    pc = axs[0].pcolormesh(1e15*CUPRAD_res.tgrid[k_t_min:k_t_max], r_grid, sym_data.T, shading='auto')
    cbar = fig.colorbar(pc, ax=axs[0], orientation = 'horizontal')
    progress_line, = axs[0].plot([], [], 'r-')  # Horizontally progressing line
    axs[0].set_xlabel(r'$t~[\mathrm{fs}]$')
    axs[0].set_ylabel(r'$\rho~[\mu\mathrm{m}]$')
    cbar.ax.set_xlabel('Intensity [harmonic cut-off]')


    plot1, = axs[1].plot(1e15*(tgrid_TDSE-0.5*tgrid_TDSE[-1])*units.TIMEau, Efields_to_plot[0])
    axs[1].set_xlabel(r'$t~[\mathrm{fs}]$')
    axs[1].set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')


    plot2, = axs[2].semilogy(Hgrid_TDSE, spectra_to_plot[0])
    axs[2].set_xlabel('harmonic order [-]')
    axs[2].set_ylabel(r'$\mathcal{E}_{\text{XUV}}~[\mathrm{arb.~u.}]$')

    axs[2].set_xlim((Hgrid_TDSE[0], Hmax_plot))

    def update(frame):
        # Update the progressing line in the pcolormesh plot
        progress_line.set_data([1e15*CUPRAD_res.tgrid[k_t_min],1e15*CUPRAD_res.tgrid[k_t_max]],
                                2*[1e6*rgrid_TDSE[frame]])
        

        plot1.set_ydata(Efields_to_plot[frame])
        plot2.set_ydata(spectra_to_plot[frame])
        
        return [progress_line, plot1, plot2] # , line1, line2]




    # Ensure the layout does not have overlaps and everything is nicely spaced
    fig.tight_layout()

    ani = matplotlib.animation.FuncAnimation(fig, update, frames=k_r_max_TDSE, blit=True)
    # ani = matplotlib.animation.FuncAnimation(fig, update, frames=3, blit=True)



plt.close(fig)
HTML(ani.to_jshtml())

This shows the laser pulse at a fixed $z$-position plane. It then show the slices of the electric field scanning the plane along the radial coordinate $\rho$. FInally, it shows the corresponding harmonic spectra.

### Hankel results
Finally, we analyse the results of the Hankel transform. First, we again define the parmeters. Note, that we use a separate file for the Hankel results. (*If you merged the results, you can modify the code to use only the main archive.*)


In [ ]:
h5_filename_Hankel = os.path.join(outputs_path,'results_Hankel.h5')

rmax = 0.007                # [m]    the radial dimension to read the data
XUV_theta_range = [-4, 4]   # [mrad] the divergence angle for plotting 
orders_to_plot = 4          # the range of the logarithmic plot of the spatially resolved harmonic spectra

H_interest = np.asarray([15, 19, 23]) # harmonics for which we show the build-up
multipliers = [1, 1, 1]               # multipliers applied to the build-up to fit the figure nicely
delta_H = 1.                          # camera spectral range around the harmonics of interest

In [ ]:
# import data & basic analyses
with h5py.File(h5_filename, 'r') as f1, h5py.File(h5_filename_Hankel, 'r') as f2:
    # load Hankel data = XUV camera
    ogrid_Hankel = f2[MMA.paths['Hankel_outputs'] + '/ogrid'][:]
    rgrid_Hankel = f2[MMA.paths['Hankel_outputs'] + '/rgrid'][:]
    zgrid_Hankel = f2[MMA.paths['Hankel_outputs'] + '/zgrid'][:]

    camera_distance = mn.readscalardataset(
        f1,
        MMA.paths['Hankel_inputs'] + '/distance_FF',
        'N'
    )

    theta_grid_Hankel = np.arctan(rgrid_Hankel / camera_distance)
    Hgrid_Hankel = ogrid_Hankel / CUPRAD_res.omega0

    kr_max = mn.FindInterval(rgrid_Hankel, rmax) + 1
    rgrid_Hankel = rgrid_Hankel[:kr_max]
    theta_grid_Hankel = theta_grid_Hankel[:kr_max]

    cumulative_field = (
        f2[MMA.paths['Hankel_outputs'] + '/cumulative_field'][:, :kr_max, :, 0]
        + 1j * f2[MMA.paths['Hankel_outputs'] + '/cumulative_field'][:, :kr_max, :, 1]
    )


# find maxima of the harmonics of interest
H_idx = [
    tuple(mn.FindInterval(
        Hgrid_Hankel,
        (H_interest[k1] - delta_H, H_interest[k1] + delta_H)
    ))
    for k1 in range(len(H_interest))
]

H_max_interest = [
    np.max(
        np.abs(cumulative_field[:, :, H_idx[k1][0]:H_idx[k1][1]]),
        axis=(1, 2)
    )
    for k1 in range(len(H_interest))
]


# Code to create the animated figure
fig = plt.figure(figsize=(14, 6))

# Define subplots using subplot2grid
ax1 = plt.subplot2grid((3, 2), (0, 0), rowspan=2)  # upper left
ax2 = plt.subplot2grid((3, 2), (0, 1), rowspan=2)  # upper right
ax3 = plt.subplot2grid((3, 2), (2, 0), colspan=2)  # bottom, spanning both columns


# Initial symmetrised data
r_grid_sym, sym_data = mn.symmetrize_y(
    rgrid_Hankel,
    np.abs(cumulative_field[0, :, :]).T
)

theta_grid_sym, sym_data = mn.symmetrize_y(
    theta_grid_Hankel,
    np.abs(cumulative_field[0, :, :]).T
)


# Upper panels: spatially resolved spectra
pc1 = ax1.pcolormesh(
    Hgrid_Hankel,
    1e3 * theta_grid_sym,
    sym_data.T,
    shading='auto'
)

pc2 = ax2.pcolormesh(
    Hgrid_Hankel,
    1e3 * theta_grid_sym,
    sym_data.T,
    shading='auto',
    norm=colors.LogNorm(
        vmin=(10**(-orders_to_plot)) * sym_data.max(),
        vmax=sym_data.max()
    )
)

ax1.set_ylim(XUV_theta_range)
ax2.set_ylim(XUV_theta_range)

ax1.set_title('spatially resolved XUV spectrum (linscale)')
ax2.set_title('spatially resolved XUV spectrum (logscale)')

ax1.set_xlabel('harmonic order [-]')
ax2.set_xlabel('harmonic order [-]')

ax1.set_ylabel(r'divergence [mrad]')

cbar1 = fig.colorbar(pc1, ax=ax1)
cbar2 = fig.colorbar(pc2, ax=ax2)
cbar2.ax.set_ylabel(r'$|\mathcal{E}_{XUV}|$ [arb.u.]', rotation=90)


# Plot lines at selected harmonic orders
for k1 in range(len(H_interest)):
    ax1.plot(
        2 * [H_interest[k1]],
        1e3 * theta_grid_sym[-1] * np.asarray([-1, 1]),
        'w:',
        alpha=0.4
    )

    ax2.plot(
        2 * [H_interest[k1]],
        1e3 * theta_grid_sym[-1] * np.asarray([-1, 1]),
        'w:',
        alpha=0.4
    )


# Keep the original intentional offset:
# cumulative_field[frame] corresponds to zgrid_Hankel[frame+1]
title = fig.suptitle(
    "z={:.2f}".format(1e3 * zgrid_Hankel[1]) + ' mm'
)


# Bottom panel: cumulative XUV signal only, no density profile
ax3.set_xlabel(r'$z~[\mathrm{mm}]$')
ax3.set_ylabel(r'XUV signal $[\mathrm{arb. u.}]$')

for k1 in range(len(H_interest)):
    if len(zgrid_Hankel) == len(H_max_interest[k1][:]):
        signal_plot = H_max_interest[k1][:]
    else:
        signal_plot = np.append(0, H_max_interest[k1][:])

    ax3.plot(
        1e3 * zgrid_Hankel,
        multipliers[k1] * signal_plot,
        label='H' + str(H_interest[k1]) + f' (x {multipliers[k1]:.1f})'
    )

ax3.legend()

# Progress indicator showing the current z-position
progress_line = ax3.axvline(
    1e3 * zgrid_Hankel[1],
    color='r',
    linestyle='-'
)


def update(frame):
    # Update the data
    data = (
        mn.symmetrize_y(
            rgrid_Hankel,
            np.abs(cumulative_field[frame, :, :]).T
        )[1]
    ).T

    pc1.set_array(data.ravel())
    pc1.set_clim(data.min(), data.max())

    pc2.set_array(data.ravel())
    pc2.set_clim(
        (10**(-orders_to_plot)) * data.max(),
        data.max()
    )

    # Intentional offset:
    # the cumulative field index is frame,
    # but the corresponding propagation position is frame+1.
    z_now = zgrid_Hankel[frame + 1]

    title.set_text(
        "z={:.2f}".format(1e3 * z_now) + ' mm'
    )

    # Update the progress indicator
    progress_line.set_xdata([1e3 * z_now, 1e3 * z_now])

    return [pc1, pc2, progress_line]


# Ensure the layout does not have overlaps and everything is nicely spaced
fig.tight_layout()

ani = matplotlib.animation.FuncAnimation(
    fig,
    update,
    frames=len(zgrid_Hankel) - 1,
    blit=True
)

plt.close(fig)

HTML(ani.to_jshtml())

This show the build-up of the harmonic spectra. As we used a very short medium, the different planes are almost identical and phase matched showing the linear growth of the XUV signal (defined as the maximum over the respective part of the XUV camera, *find in the code this computation of `H_max_interest`*). Note that the cummulative signal is available only if this is enabled by the input. Otherwise, only the final transform is available in the dataset `FF_integrated`. (*Inspect the HDF5 file to see it directly.*)

### Thank you for reaching the end of the tutorial and happy computing!